In [3]:
import math

In [9]:
thetaOri = []
kOri = []

for n in range(20):
    thetaOri.append(math.atan(1/(2**n)))#生成tan^-1(1/2^(n))的队列
    kOri.append(1/math.sqrt(1+(2**(-2*n))))#生成1/(sqrt(1+2^(-2n)))))的队列

thetaInt = []
kInt = []
for n in range(20):
    thetaInt.append(thetaOri[n]*(2**20))#缩放到2的20次方
    kInt.append(kOri[n]*(2**20))#缩放到2的20次方

print(thetaOri)
print(thetaInt)
print(kOri)
print(kInt)

[0.7853981633974483, 0.4636476090008061, 0.24497866312686414, 0.12435499454676144, 0.06241880999595735, 0.031239833430268277, 0.015623728620476831, 0.007812341060101111, 0.0039062301319669718, 0.0019531225164788188, 0.0009765621895593195, 0.0004882812111948983, 0.00024414062014936177, 0.00012207031189367021, 6.103515617420877e-05, 3.0517578115526096e-05, 1.5258789061315762e-05, 7.62939453110197e-06, 3.814697265606496e-06, 1.907348632810187e-06]
[823549.6645826427, 486169.75525562925, 256878.7466669147, 130395.66276186492, 65450.866110320974, 32757.33957897699, 16382.666861945114, 8191.833339436583, 4095.9791668573994, 2047.9973958392939, 1023.999674479353, 511.99995930990167, 255.99999491373717, 127.99999936421713, 63.99999992052714, 31.99999999006589, 15.999999998758236, 7.999999999844779, 3.9999999999805973, 1.9999999999975746]
[0.7071067811865475, 0.8944271909999159, 0.9701425001453319, 0.9922778767136677, 0.9980525784828885, 0.9995120760870788, 0.9998779520346953, 0.999969483818787

In [10]:
kk = 1
for i in kOri:
    kk = kk * i
#模长缩放因数kk：假设确定旋转20次，将伪旋转操作的模修正到正确结果的模的模长缩放因数应该是每一次伪旋转的缩放因数的积
kk

0.6072529350092495

In [14]:
thetaIntF = []

for i in thetaOri:
    thetaIntF.append(i/math.pi * 2**20)
    #将thetaOri的单位从弧度制转换到角度制(2pi->2*2*20)

In [17]:
def cordicIntPt4(xin):

    print("input a is {}, actual Radix is {}".format(xin, xin/(2.**20)))
    if (xin < 0):#若输入的xin小于0，则加上2*pi调整到正值
        xfix = xin + 2*2**20  # -> add at 21 bit
    else :
        xfix = xin
    if 0 <= xfix < 524288:#[0,0.5*pi)
        a = xfix#a从0开始+xfix
        sgnS = 1#sin结果的符号，1:+
        sgnC = 1#cos结果的符号，1:+
    elif 524288 <= xfix < 1048576:#[0.5*pi,pi)
        a = 1048576 - xfix#相当于第一象限关于y轴镜像翻转，从pi开始-xfix
        sgnS = 1#sin结果的符号，1:+
        sgnC = -1#cos结果的符号，-1:-
    elif 1048576 <= xfix < 1572864:#[pi,1.5*pi)
        a = xfix - 1048578#相当于第二象限关于x轴镜像翻转，从pi开始+xfix
        sgnS = -1#sin结果的符号，-1:-
        sgnC = -1#cos结果的符号，-1:-
    else :#[1.5*pi,2*pi)
        a = 2097152 - xfix#相当于第一象限关于x轴镜像翻转，从2pi开始-xfix
        sgnS = -1#sin结果的符号，-1:-
        sgnC = 1#cos结果的符号，1:+

    x = int(kk * 2 ** 20)#从(x,y)=(1,0)开始，乘以 模长缩放因数(同样量化到2^20)
    y = 0
    angleNew = 0#angle初始值从0开始
    angleRm = a-angleNew#需要转动到的角度a和当前的角度angleNew的差，a的值在前面的分类讨论中已经确定

    for i in range(20):#转动20次
        if angleRm > 0:#如果需要正向旋转（逆时针）
            angleNew = angleNew + int(thetaIntF[i])#角度指针angleNew加上当前迭代对应的转动的角度thetaIntF[i]
            angleRm = a - angleNew#更新输入指定的转动角度a和角度指针angleNew的差
            xTmp = x - (y >> i)#x做伪旋转
            yTmp = y + (x >> i)#y做伪旋转
            x = xTmp#更新x的值
            y = yTmp#更新y的值
            dir = 0#旋转方向，0:正，1:反
        else:#如果需要反向旋转（顺时针）
            angleNew = angleNew - int(thetaIntF[i])#角度指针减去当前迭代对应的转动的角度
            angleRm = a - angleNew#更新输入指定的转动角度a和角度指针angleNew的差
            xTmp = x + (y >> i)#x做伪旋转
            yTmp = y - (x >> i)#y做伪旋转
            x = xTmp#更新x的值
            y = yTmp#更新y的值
            dir = 1#旋转方向，0:正，1:反
        #显示当前迭代的的简要信息
        print(
            "{}th rotation, angleNew is {}, angleRm is {}, xTmp is {}, yTmp is {}, direction is {}".format(i, angleNew,
                                                                                                           angleRm,
                                                                                                           xTmp, yTmp,
                                                                                                          dir))
    #最后将结果修正到预先决定好的象限
    xpred = sgnC * (x/(2.**20))
    ypred = sgnS * (y/(2.**20))
    #显示最终的结果和误差
    print("result: cos is {}, sin is {}".format(sgnC*x, sgnS*y))
    print("cos gt is {}, get {}, err is {} \nsin gt is {}, get {}, err is {}".format(math.cos(xin / (2. ** 20) * math.pi),
                                                                                     xpred,
                                                                                     math.cos(xin / (2. ** 20) * math.pi) - xpred,
                                                                                     math.sin(xin / (2. ** 20) * math.pi),
                                                                                     ypred,
                                                                                     math.sin(xin / (2. ** 20) * math.pi) - ypred))

In [16]:
cordicIntPt4(500000)

input a is 500000, actual Radix is 0.476837158203125
0th rotation, angleNew is 262144, angleRm is 237856, xTmp is 636750, yTmp is 636750, direction is 0
1th rotation, angleNew is 416896, angleRm is 83104, xTmp is 318375, yTmp is 955125, direction is 0
2th rotation, angleNew is 498663, angleRm is 1337, xTmp is 79594, yTmp is 1034718, direction is 0
3th rotation, angleNew is 540169, angleRm is -40169, xTmp is -49745, yTmp is 1044667, direction is 0
4th rotation, angleNew is 519336, angleRm is -19336, xTmp is 15546, yTmp is 1047777, direction is 1
5th rotation, angleNew is 508910, angleRm is -8910, xTmp is 48289, yTmp is 1047292, direction is 1
6th rotation, angleNew is 503696, angleRm is -3696, xTmp is 64652, yTmp is 1046538, direction is 1
7th rotation, angleNew is 501089, angleRm is -1089, xTmp is 72828, yTmp is 1046033, direction is 1
8th rotation, angleNew is 499786, angleRm is 214, xTmp is 76914, yTmp is 1045749, direction is 1
9th rotation, angleNew is 500437, angleRm is -437, xTmp

In [11]:
rotate = 20

In [28]:
#这个cordicIntPt和上面的整体流程差不多，区别在于它可以现场决定旋转的次数以及量化的精度
def cordicIntPt(xin, rotate, xfrac):
    #生成tan^-1(1/2^(n))的队列：thetaOri
    #生成k的队列：kOri
    thetaOri = []
    kOri = []
    for i in range(rotate):
        thetaOri.append(math.atan(1/(2**i)))
        kOri.append(1/math.sqrt(1+(2**(-2*i))))
    #生成整数化的theta和k
    thetaInt = []
    kInt = []
    for i in range(rotate):
        thetaInt.append(thetaOri[i]*2**xfrac)
        kInt.append(kOri[i]*2**xfrac)
    #生成模修正因子kk
    kk = 1
    for i in kOri:
        kk = kk * i
    #转换到角度制
    thetaIntF = []
    for i in thetaOri:
        thetaIntF.append(i/math.pi * 2**xfrac)
    

    print("input a is {}, actual Radix is {}".format(xin, xin/(2.**xfrac)))
    #如果xin是负数则加上一个周期修正到正数
    if (xin < 0):
        xfix = xin + 2*2**xfrac  # -> add at 21 bit
    else :
        xfix = xin
    #讨论四个象限的情况，分别给出目标旋转角度a；sin的符号修正，cos的符号修正
    if 0 <= xfix < 2**(xfrac-1):
        a = xfix
        sgnS = 1
        sgnC = 1
    elif 2**(xfrac-1) <= xfix < 2**(xfrac):
        a = 2**xfrac - xfix
        sgnS = 1
        sgnC = -1
    elif 2**(xfrac) <= xfix < 2**(xfrac+1):
        a = xfix - 2**(xfrac)
        sgnS = -1
        sgnC = -1
    else :
        a = 2**(xfrac+1) - xfix
        sgnS = -1
        sgnC = 1

    x = int(kk * 2 ** xfrac)
    y = 0
    angleNew = 0
    angleRm = a

    for i in range(rotate):
        if angleRm > 0:
            angleNew = angleNew + int(thetaIntF[i])
            angleRm = a - angleNew
            xTmp = x - (y >> i)
            yTmp = y + (x >> i)
            x = xTmp
            y = yTmp
            dir = 0
        else:
            angleNew = angleNew - int(thetaIntF[i])
            angleRm = a - angleNew
            xTmp = x + (y >> i)
            yTmp = y - (x >> i)
            x = xTmp
            y = yTmp
            dir = 1
        # print(
        #     "{}th rotation, angleNew is {}, angleRm is {}, xTmp is {}, yTmp is {}, direction is {}".format(i, angleNew,
        #                                                                                                    angleRm,
        #                                                                                                    xTmp, yTmp,
        #                                                                                                    dir))
    xpred = sgnC * (x/(2.**xfrac))
    ypred = sgnS * (y/(2.**xfrac))
    print("result: cos is {}, sin is {}".format(sgnC*x, sgnS*y))
    print("cos int is {}, err is {} \n sin int is {}, err is {}".format(
        math.cos(xin/(2.**xfrac)*math.pi)*(2.**xfrac), 
        math.cos(xin/(2.**xfrac)*math.pi)*(2.**xfrac) - sgnC*x, 
        math.sin(xin/(2.**xfrac)*math.pi)*(2.**xfrac), 
        math.sin(xin/(2.**xfrac)*math.pi)*(2.**xfrac) - sgnS*y, 

    ))
    # print("cos gt is {}, get {}, err is {} \nsin gt is {}, get {}, err is {}".format(math.cos(xin / (2. ** xfrac) * math.pi),
    #                                                                                  xpred,
    #                                                                                  math.cos(xin / (2. ** xfrac) * math.pi) - xpred,
    #                                                                                  math.sin(xin / (2. ** xfrac) * math.pi),
    #                                                                                  ypred,
    #                                                                                  math.sin(xin / (2. ** xfrac) * math.pi) - ypred))

In [31]:
cordicIntPt(500000, 20, 20)

input a is 500000, actual Radix is 0.476837158203125
result: cos is 76247, sin is 1045799
cos int is 76235.68008989406, err is -11.319910105943563 
 sin int is 1045801.008250246, err is 2.008250246057287


In [30]:
cordicIntPt(500000, 14, 20)

input a is 500000, actual Radix is 0.476837158203125
result: cos is 76275, sin is 1045798
cos int is 76235.68008989406, err is -39.31991010594356 
 sin int is 1045801.008250246, err is 3.008250246057287


In [35]:
cordicIntPt(1/2*2**14, 14, 14)

input a is 8192.0, actual Radix is 0.5
result: cos is -2, sin is 16383
cos int is 1.0032306578615117e-12, err is 2.000000000001003 
 sin int is 16384.0, err is 1.0


In [33]:
math.cos(1/2*math.pi)

6.123233995736766e-17